# Jina Reranker — 긴 문서 처리에 강한 재순위화

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Jina Reranker**의 차별화 특징(최대 8K 토큰 지원)을 이해하기
- `JinaRerank` 클래스로 재순위화 파이프라인 구현하기
- Cross-Encoder, Cohere, Jina 세 가지 Reranker를 비교해 적절한 선택 기준 파악하기

## 사전 지식

- 01-Cross-Encoder-Reranker, 02-Cohere-Reranker의 Two-Stage Retrieval 개념
- Jina API 키 발급 방법 (`JINA_API_KEY` 환경변수 필요)

---

## Reranker 3종 비교

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> VR[벡터 검색<br/>OpenAI + FAISS<br/>k=10]:::process
    VR --> JR[Jina Rerank API<br/>최대 8K 토큰<br/>top_n=3]:::external
    JR --> R[관련성 점수 포함<br/>상위 3개 문서]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

| 특징 | Jina Reranker | Cohere Rerank | Cross-Encoder |
|------|:---:|:---:|:---:|
| 최대 토큰 | **8K** | 4K | 512 |
| 속도 | 빠름 | 빠름 | 보통 |
| 비용 | $ | $$ | 무료 |
| 추천 용도 | **긴 문서** | 프로덕션 | 개발·테스트 |

> **실무 팁**: 논문, 보고서, 기술 문서처럼 청크 크기가 2000자 이상인 경우 Jina Reranker가 잘려나가는 현상 없이 전체를 분석할 수 있어요.

> **실무 팁**: 3개 Reranker 모두 `relevance_score`를 `metadata`에 추가합니다. 이 점수를 활용해 "관련성이 특정 임계값 이하인 문서는 LLM에 전달하지 않는" 추가 필터링도 가능해요.

## 1. 환경 설정

### 1.1 Jina API 키 발급

1. [Jina AI 웹사이트](https://jina.ai/reranker)에서 API 키 발급
2. `.env` 파일에 API 키 추가:

```bash
JINA_API_KEY="your-jina-api-key"
```

### 1.2 Jina Reranker 모델

- `jina-reranker-v2-base-multilingual`: 다국어 기본 모델 (권장)
- `jina-reranker-v1-base-en`: 영어 특화 모델
- `jina-reranker-v1-turbo-en`: 빠른 속도 (영어)

In [ ]:
# 필요한 라이브러리 import
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import JinaRerank

# 환경 변수 로드

# 아래에 코드를 작성하세요


## 2. 문서 출력 도우미 함수

In [ ]:
def pretty_print_docs(docs, show_scores=True):
    """검색된 문서를 보기 좋게 출력하는 함수"""
    print(f"\n{'=' * 100}")
    print(f"총 {len(docs)}개 문서 검색됨")
    print(f"{'=' * 100}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        
        # Jina Reranker도 relevance_score를 metadata에 추가
        if show_scores and 'relevance_score' in doc.metadata:
            score = doc.metadata['relevance_score']
            print(f"관련성 점수: {score:.4f}")
        
        print(f"내용: {doc.page_content}")
        print(f"{'─' * 100}\n")


## 3. 기본 검색 시스템 구축

In [ ]:
# 1단계: 문서 로드

# 아래에 코드를 작성하세요


In [ ]:
# 2단계: 텍스트 분할

# 아래에 코드를 작성하세요


In [ ]:
# 3단계: 벡터스토어 생성 (OpenAI 임베딩 사용)

# 아래에 코드를 작성하세요


In [ ]:
# ---------------------------------------------------
# Reranker 적용 전 기본 벡터 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: base_retriever로 쿼리를 검색하고 결과를 출력하세요
# 힌트: invoke(query) 호출 후 pretty_print_docs(docs, show_scores=False)
# 예상 결과: 10개 문서가 OpenAI 임베딩 유사도 순으로 출력됨
# ============================================================

# 기본 검색 테스트


# Reranker 적용 전 기본 검색


## 4. Jina Reranker 적용

Jina AI의 Rerank API를 사용하여 검색 결과를 재정렬합니다.

In [ ]:
# ---------------------------------------------------
# Jina Reranker 설정 — 긴 문서 처리 능력 활용
# ---------------------------------------------------
# ============================================================
# TODO: JinaRerank와 ContextualCompressionRetriever로 재정렬 파이프라인을 만드세요
# 힌트: JinaRerank(model="jina-reranker-v2-base-multilingual", top_n=3)
# 예상 결과: "Jina Reranker 설정 완료" 출력 (최대 8K 토큰 지원 표시)
# ============================================================

    # 1단계: Jina Reranker 초기화
    # jina-reranker-v2-base-multilingual: 최대 8K 토큰 지원 다국어 Reranker

    # 2단계: 압축 검색기 생성


In [ ]:
# ---------------------------------------------------
# Reranker 적용 후 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: compression_retriever로 같은 쿼리를 검색하고 관련성 점수를 확인하세요
# 힌트: invoke(query) 후 pretty_print_docs(docs, show_scores=True)
# 예상 결과: 3개 문서와 Jina 관련성 점수 출력
# ============================================================

    # Reranker 적용 검색


## 5. 결과 비교 및 분석

In [ ]:
# ---------------------------------------------------
# Jina Reranker 적용 전후 결과 비교 분석
# ---------------------------------------------------

# 아래에 코드를 작성하세요


## 6. 다양한 쿼리 테스트

In [ ]:
# ---------------------------------------------------
# 다양한 쿼리로 Jina Reranker 성능 검증
# ---------------------------------------------------
# ============================================================
# TODO: 여러 쿼리를 순회하며 Jina Reranker로 검색하고 관련성 점수를 확인하세요
# 힌트: for 루프로 test_queries 순회, compression_retriever.invoke(test_query)
# 예상 결과: 각 쿼리별 상위 3개 문서와 Jina 관련성 점수 출력
# ============================================================

    # 다양한 쿼리로 Jina Reranker 성능 검증


        
        # Jina Reranker 적용


## 7. 핵심 정리

### Reranker 비교

| 특징 | Jina Reranker | Cohere Rerank | Cross-Encoder (로컬) |
|------|:---:|:---:|:---:|
| 최대 토큰 | **8K** | 4K | 512 |
| 속도 | 빠름 | 빠름 | 보통 |
| 정확도 | 높음 | 최고 수준 | 높음 |
| 비용 | $ | $$ | 무료 |
| 추천 용도 | **긴 문서** | 프로덕션 | 개발/테스트 |

### 기본 사용법

```python
from langchain_community.document_compressors import JinaRerank

compressor = JinaRerank(
    model="jina-reranker-v2-base-multilingual",
    top_n=3,
)
```

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | Reranker 모델명 | `jina-reranker-v2-base-multilingual` |
| `top_n` | 최종 반환 문서 수 | 3 |
| `jina_api_key` | Jina API 키 | 환경변수 `JINA_API_KEY` |

## 마무리 정리 — Reranker 시리즈 완성

이 시리즈(01~03)에서 배운 세 가지 Reranker를 정리해요:

| Reranker | 환경 | 토큰 한도 | 비용 | 추천 상황 |
|---------|------|:---:|:---:|----------|
| **Cross-Encoder** | 로컬 | 512 | 무료 | 개발·테스트, 데이터 보안 |
| **Cohere** | 클라우드 | 4K | $$ | 한국어 프로덕션 |
| **Jina** | 클라우드 | **8K** | $ | 긴 문서, 비용 효율 |

### 이제 무엇을 배우나요?

다음 6-7 RAG Process 섹션에서는 지금까지 배운 Retriever, Reranker를 모두 조합해 완전한 End-to-End RAG 시스템을 구축해요. PDF, 웹, 대화형, RAPTOR 등 다양한 시나리오를 직접 구현해봅니다.